# Run All Four Optimizer Modes

This notebook trains the same model in four modes and reports final train/test MSE for each mode.

## Step 1: Imports
Import libraries for data prep, modeling, and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="Unknown extension is not supported and will be removed")


## Step 2: Rename, Clean, and Impute
Load the dataset, rename columns, clean invalid values, and impute missing feature values.


In [ ]:
# Load and clean data
df = pd.read_excel(
    "https://raw.githubusercontent.com/brian-fischer/DATA-5100/main/EdGap_data.xlsx",
    dtype={"NCESSCH School ID": object},
)

df = df.rename(columns={
    "NCESSCH School ID": "id",
    "CT Pct Adults with College Degree": "percent_college",
    "CT Unemployment Rate": "rate_unemployment",
    "CT Pct Childre In Married Couple Family": "percent_married",
    "CT Median Household Income": "median_income",
    "School ACT average (or equivalent if SAT score)": "average_act",
    "School Pct Free and Reduced Lunch": "percent_lunch",
})

df.loc[df["average_act"] < 1, "average_act"] = np.nan
df.loc[df["percent_lunch"] < 0, "percent_lunch"] = np.nan

input_modes = df[df.columns.difference(["id", "average_act"])].mode().iloc[0]
df = df.fillna(input_modes)
df = df.dropna()

X = df[df.columns.difference(["id", "average_act"]) ]
y = df["average_act"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4242)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

X_train_j = jnp.array(X_train)
X_test_j = jnp.array(X_test)
y_train_j = jnp.array(y_train.to_numpy()).reshape(-1, 1)
y_test_j = jnp.array(y_test.to_numpy()).reshape(-1, 1)


## Step 3: Model Setup and Training Utilities (JAX)
Define architecture, loss, gradients, and update rules for SGD and Momentum.


In [ ]:
# Shared settings
epochs = 1000
seed = 0

mini_batch_size = 64
momentum_batch_size = 128

sgd_lr = 5e-4
mini_gd_lr = 5e-4
momentum_lr = 2e-4
momentum_beta = 0.9

layer_sizes = [X_train.shape[1], 64, 32, 1]

def init_network_params(layer_sizes, seed=42):
    key = jax.random.PRNGKey(seed)
    params = []
    for idx in range(len(layer_sizes) - 1):
        key, subkey = jax.random.split(key)
        W = jax.random.normal(subkey, (layer_sizes[idx], layer_sizes[idx + 1])) * 0.01
        b = jnp.zeros((1, layer_sizes[idx + 1]))
        params.append({"W": W, "b": b})
    return params

def relu(x):
    return jnp.maximum(0, x)

def forward(params, X):
    activations = X
    num_layers = len(params)
    for layer_idx, layer in enumerate(params):
        z = activations @ layer["W"] + layer["b"]
        if layer_idx < num_layers - 1:
            activations = relu(z)
        else:
            activations = z
    return activations

def mse_loss(params, X, y):
    y_pred = forward(params, X)
    return jnp.mean((y_pred - y) ** 2)

grad_fn = jax.jit(jax.grad(mse_loss))

def sgd_update(params, grads, lr):
    return [
        {
            "W": layer["W"] - lr * grad["W"],
            "b": layer["b"] - lr * grad["b"],
        }
        for layer, grad in zip(params, grads)
    ]

def init_velocities(params):
    return [
        {
            "W": jnp.zeros_like(layer["W"]),
            "b": jnp.zeros_like(layer["b"]),
        }
        for layer in params
    ]

def momentum_update(params, grads, velocities, lr, beta):
    new_params = []
    new_velocities = []
    for layer, grad, vel in zip(params, grads, velocities):
        vW = beta * vel["W"] + grad["W"]
        vb = beta * vel["b"] + grad["b"]
        new_params.append({"W": layer["W"] - lr * vW, "b": layer["b"] - lr * vb})
        new_velocities.append({"W": vW, "b": vb})
    return new_params, new_velocities


## Step 4: Run All Four Optimizer Modes
Train with all four modes and report final train/test MSE for each mode.


In [ ]:
mode_configs = [
    ("SGD (Full Batch)", "sgd", "mini"),
    ("Mini-Batch GD", "mini_gd", "mini"),
    ("Momentum (Full Batch)", "momentum", "full"),
    ("Momentum (Mini-Batch)", "momentum", "mini"),
]

results = []
mode_histories = {}

for mode_name, optimizer_mode_local, momentum_batch_style_local in mode_configs:
    params_local = init_network_params(layer_sizes, seed=2999)

    if optimizer_mode_local == "sgd":
        lr_local = sgd_lr
        batch_size_local = X_train.shape[0]
        shuffle_local = False
        use_momentum_local = False
    elif optimizer_mode_local == "mini_gd":
        lr_local = mini_gd_lr
        batch_size_local = mini_batch_size
        shuffle_local = True
        use_momentum_local = False
    else:
        lr_local = momentum_lr
        use_momentum_local = True
        if momentum_batch_style_local == "full":
            batch_size_local = X_train.shape[0]
            shuffle_local = False
        else:
            batch_size_local = momentum_batch_size
            shuffle_local = True

    n_local = X_train_j.shape[0]
    is_full_batch_local = (batch_size_local == X_train_j.shape[0])
    rng_local = jax.random.PRNGKey(seed)
    velocities_local = init_velocities(params_local) if use_momentum_local else None

    train_history_local = np.zeros(epochs, dtype=float)
    test_history_local = np.zeros(epochs, dtype=float)

    for epoch in range(epochs):
        if is_full_batch_local:
            grads_local = grad_fn(params_local, X_train_j, y_train_j)
            if use_momentum_local:
                params_local, velocities_local = momentum_update(
                    params_local, grads_local, velocities_local, lr_local, momentum_beta
                )
            else:
                params_local = sgd_update(params_local, grads_local, lr_local)
        else:
            if shuffle_local:
                rng_local, perm_key_local = jax.random.split(rng_local)
                perm_local = jax.random.permutation(perm_key_local, n_local)
                X_epoch_local = X_train_j[perm_local]
                y_epoch_local = y_train_j[perm_local]
            else:
                X_epoch_local = X_train_j
                y_epoch_local = y_train_j

            for start_local in range(0, n_local, batch_size_local):
                X_batch_local = X_epoch_local[start_local:start_local + batch_size_local]
                y_batch_local = y_epoch_local[start_local:start_local + batch_size_local]
                grads_local = grad_fn(params_local, X_batch_local, y_batch_local)
                if use_momentum_local:
                    params_local, velocities_local = momentum_update(
                        params_local, grads_local, velocities_local, lr_local, momentum_beta
                    )
                else:
                    params_local = sgd_update(params_local, grads_local, lr_local)

        train_history_local[epoch] = float(mse_loss(params_local, X_train_j, y_train_j))
        test_history_local[epoch] = float(mse_loss(params_local, X_test_j, y_test_j))

    mode_histories[mode_name] = {
        "train": train_history_local,
        "test": test_history_local,
    }

    results.append({
        "Mode": mode_name,
        "Final Train MSE": float(train_history_local[-1]),
        "Final Test MSE": float(test_history_local[-1]),
    })

## Step 4.5: Print Architecture and Optimizer Results

In [ ]:
results_df = pd.DataFrame(results)
print(f"Architecture: {layer_sizes}")
print()
print(results_df.to_string(index=False))
best_row = results_df.loc[results_df["Final Test MSE"].idxmin()]
print(f"Best performing method: {best_row['Mode']} (Final Test MSE: {best_row['Final Test MSE']:.4f})")


## Step 5: Plot Training Curves by Mode
Plot Train Loss vs Epoch for each of the four optimizer modes.


In [ ]:
# first epoch index that is not an extreme spike (based on train losses across modes)
all_train_hist = np.array([mode_histories[name]["train"] for name, _, _ in mode_configs])
median_curve = np.median(all_train_hist, axis=0)
later_med = np.median(median_curve[10:]) if epochs > 10 else np.median(median_curve)
threshold = 3.0 * later_med
valid_idx = np.where(median_curve <= threshold)[0]
start_epoch = int(valid_idx[0]) if len(valid_idx) else 0

epoch_idx = np.arange(start_epoch, epochs)
mode_names = [name for name, _, _ in mode_configs]

for mode_name in mode_names:
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot()

    train_curve = mode_histories[mode_name]["train"][start_epoch:]

    ax.plot(epoch_idx, train_curve, label="Train", color="tab:blue")
    ax.set_title(f"{mode_name}: Train Loss vs Epoch\nFinal Train MSE: {train_curve[-1]:.4f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train MSE")
    ax.set_ylim(train_curve.min(), train_curve.max())
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
